In [2]:
import requests
from bs4 import BeautifulSoup
import re
import time
import spacy

In [3]:
def get_urls_from_file(filename):
    urls = []
    with open(filename, 'r', encoding='utf-8') as file:
        for line in file:
            match = re.search(r'(https?://\S+)', line)
            if match:
                urls.append(match.group(1))

    return urls

In [4]:
pl_urls = get_urls_from_file('pl.txt')
en_urls = get_urls_from_file('en.txt')

In [5]:
def fetch_wikipedia_text(url):
    headers = {
        'User-Agent': 'ProjektAnalizyTekstuNLP/1.0 (kontakt_testowy@example.com)'
    }
    
    response = requests.get(url, headers=headers)
    response.raise_for_status() 
    
    soup = BeautifulSoup(response.text, 'html.parser')
    paragraphs = soup.find_all('p')
    
    text = ' '.join([p.get_text().strip() for p in paragraphs])
    return text

In [6]:
pl_corpus_raw = []

for i, url in enumerate(pl_urls):
    print(f"Pobieranie ({i+1}/{len(pl_urls)}): {url}")
    
    try:
        text = fetch_wikipedia_text(url)
        pl_corpus_raw.append(text)
    except Exception as e:
        print(f" -> Błąd pobierania: {e}")
    
    if i < len(pl_urls) - 1:
        time.sleep(5)


Pobieranie (1/8): https://pl.wikipedia.org/wiki/Jagiellonia_Bia%C5%82ystok
Pobieranie (2/8): https://pl.wikipedia.org/wiki/Jes%C3%BAs_Imaz
Pobieranie (3/8): https://pl.wikipedia.org/wiki/Superpozycja
Pobieranie (4/8): https://pl.wikipedia.org/wiki/Mechanika_kwantowa
Pobieranie (5/8): https://pl.wikipedia.org/wiki/Funkcja_falowa
Pobieranie (6/8): https://pl.wikipedia.org/wiki/Taras_Romanczuk
Pobieranie (7/8): https://pl.wikipedia.org/wiki/Afimico_Pululu
Pobieranie (8/8): https://pl.wikipedia.org/wiki/Adrian_Siemieniec


In [7]:
en_corpus_raw = []

for i, url in enumerate(en_urls):
    print(f"Pobieranie ({i+1}/{len(en_urls)}): {url}")
    
    try:
        text = fetch_wikipedia_text(url)
        en_corpus_raw.append(text)
    except Exception as e:
        print(f" -> Błąd pobierania: {e}")
    
    if i < len(en_urls) - 1:
        time.sleep(5)

Pobieranie (1/8): https://en.wikipedia.org/wiki/Jagiellonia_Bia%C5%82ystok
Pobieranie (2/8): https://en.wikipedia.org/wiki/Jes%C3%BAs_Imaz
Pobieranie (3/8): https://en.wikipedia.org/wiki/Quantum_superposition
Pobieranie (4/8): https://en.wikipedia.org/wiki/Quantum_mechanics
Pobieranie (5/8): https://en.wikipedia.org/wiki/Wave_function
Pobieranie (6/8): https://en.wikipedia.org/wiki/Taras_Romanczuk
Pobieranie (7/8): https://en.wikipedia.org/wiki/Afimico_Pululu
Pobieranie (8/8): https://en.wikipedia.org/wiki/Adrian_Siemieniec


In [9]:
nlp_pl = spacy.load("pl_core_news_sm", disable=['ner', 'parser'])
nlp_en = spacy.load("en_core_web_sm", disable=['ner', 'parser'])

In [14]:
def preprocess_text(text, nlp_model, lang='pl'):
    text_cleaned = re.sub(r'[^a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ]', ' ', text)
    
    doc = nlp_model(text_cleaned.lower())
    processed_tokens = []
    
    for token in doc:
        if not token.is_stop and not token.is_space and len(token.text) > 1:
            processed_tokens.append(token.lemma_)
            
    return processed_tokens


In [15]:
pl_corpus_processed = [preprocess_text(text, nlp_pl, lang='pl') for text in pl_corpus_raw]
en_corpus_processed = [preprocess_text(text, nlp_en, lang='en') for text in en_corpus_raw]